# Vendor Skill Update Checker

**Purpose:** Check the vendored skills in this repo against their upstream sources and (optionally) pull updates.

**Outputs:**
- A status table comparing each skill's pinned commit to the latest upstream commit on its source path.
- An `update_skill(name, apply=True)` helper that re-downloads a skill folder from upstream and re-pins `vendor-skills.json`.

**Single source of truth:** [`vendor-skills.json`](../vendor-skills.json) at the repo root. This notebook never hardcodes skill metadata.

**GitHub rate limits:** Unauthenticated requests are limited to 60/hour. Set a `GITHUB_TOKEN` environment variable (a classic PAT with `public_repo` scope, or a fine-grained read-only token) to raise this to 5,000/hour.

## Setup & Config

In [ ]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
import requests

GITHUB_API = "https://api.github.com"


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward from the notebook's CWD until vendor-skills.json is found."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "vendor-skills.json").exists():
            return candidate
    raise FileNotFoundError(
        "vendor-skills.json not found in CWD or any parent. "
        "Run this notebook from inside the skills repo."
    )


REPO_ROOT = find_repo_root()
MANIFEST_PATH = REPO_ROOT / "vendor-skills.json"


def load_manifest() -> dict:
    return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def save_manifest(manifest: dict) -> None:
    MANIFEST_PATH.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )


_token = os.environ.get("GITHUB_TOKEN")
SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/vnd.github+json", "User-Agent": "vendor-skill-updater"})
if _token:
    SESSION.headers["Authorization"] = f"Bearer {_token}"
    print("Using authenticated GitHub requests (5,000/hr).")
else:
    print("No GITHUB_TOKEN set - using unauthenticated requests (60/hr).")

manifest = load_manifest()
print(f"Repo root: {REPO_ROOT}")
print(f"Tracked vendored skills: {len(manifest['skills'])}")

## Check for updates

For each skill, asks the GitHub API for the most recent commit that touched its `source_path` on the tracked `branch`, then compares that to the `pinned_commit` recorded in the manifest.

In [ ]:
def get_latest_commit(repo: str, path: str, branch: str) -> dict:
    """Return {sha, date, html_url} for the latest commit touching `path` on `branch`."""
    resp = SESSION.get(
        f"{GITHUB_API}/repos/{repo}/commits",
        params={"path": path, "sha": branch, "per_page": 1},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return {"sha": None, "date": None, "html_url": None}
    c = data[0]
    return {
        "sha": c["sha"],
        "date": c["commit"]["committer"]["date"],
        "html_url": c["html_url"],
    }


def check_updates(manifest: dict) -> pd.DataFrame:
    rows = []
    for s in manifest["skills"]:
        try:
            latest = get_latest_commit(s["repo"], s["source_path"], s["branch"])
            if latest["sha"] is None:
                status = "⚠ path not found upstream"
            elif latest["sha"] == s["pinned_commit"]:
                status = "✓ up to date"
            else:
                status = "⬆ update available"
        except requests.HTTPError as e:
            latest = {"sha": None, "date": None, "html_url": None}
            status = f"error: {e.response.status_code}"
        rows.append(
            {
                "skill": s["name"],
                "repo": s["repo"],
                "pinned": s["pinned_commit"][:7],
                "latest": (latest["sha"] or "")[:7],
                "latest_date": latest["date"],
                "status": status,
                "compare": (
                    f"https://github.com/{s['repo']}/compare/{s['pinned_commit'][:7]}...{latest['sha'][:7]}"
                    if latest["sha"] and latest["sha"] != s["pinned_commit"]
                    else ""
                ),
            }
        )
    return pd.DataFrame(rows)


report = check_updates(manifest)
report

## Update a skill

`update_skill(name, apply=False)` is a **dry run by default** - it reports what it would do. Pass `apply=True` to:
1. Re-download every file under the skill's `source_path` from the latest upstream commit.
2. Replace the local skill folder with the fresh copy.
3. Re-pin `vendor-skills.json` to the new commit.

Review the resulting `git diff` before committing - upstream skills can change structure or licensing.

In [ ]:
def _download_tree(repo: str, path: str, ref: str, dest: Path) -> int:
    """Recursively download a directory from GitHub into `dest`. Returns file count."""
    resp = SESSION.get(
        f"{GITHUB_API}/repos/{repo}/contents/{path}",
        params={"ref": ref},
        timeout=30,
    )
    resp.raise_for_status()
    items = resp.json()
    if isinstance(items, dict):  # a single file was requested
        items = [items]
    count = 0
    for item in items:
        if item["type"] == "dir":
            count += _download_tree(repo, item["path"], ref, dest / item["name"])
        elif item["type"] == "file":
            dest.mkdir(parents=True, exist_ok=True)
            blob = SESSION.get(item["download_url"], timeout=30)
            blob.raise_for_status()
            (dest / item["name"]).write_bytes(blob.content)
            count += 1
    return count


def update_skill(name: str, apply: bool = False) -> None:
    manifest = load_manifest()
    entry = next((s for s in manifest["skills"] if s["name"] == name), None)
    if entry is None:
        raise ValueError(f"{name!r} is not in vendor-skills.json")

    latest = get_latest_commit(entry["repo"], entry["source_path"], entry["branch"])
    if latest["sha"] is None:
        print(f"⚠ source path not found upstream for {name}; aborting.")
        return
    if latest["sha"] == entry["pinned_commit"]:
        print(f"✓ {name} already at latest ({latest['sha'][:7]}). Nothing to do.")
        return

    local = REPO_ROOT / entry["local_path"]
    print(f"{name}: {entry['pinned_commit'][:7]} -> {latest['sha'][:7]} ({latest['date']})")
    print(f"  source: {entry['repo']}/{entry['source_path']}")
    print(f"  local:  {local}")
    if not apply:
        print("  DRY RUN - pass apply=True to download and re-pin.")
        return

    staging = local.with_name(local.name + ".new")
    if staging.exists():
        shutil.rmtree(staging)
    n = _download_tree(entry["repo"], entry["source_path"], latest["sha"], staging)
    if local.exists():
        shutil.rmtree(local)
    staging.rename(local)
    entry["pinned_commit"] = latest["sha"]
    entry["pinned_ref"] = latest["sha"][:7]
    save_manifest(manifest)
    print(f"  ✓ wrote {n} file(s) and re-pinned manifest to {latest['sha'][:7]}.")
    print("  Review `git diff` before committing.")


# Examples (dry run):
for _name in report.loc[report["status"].str.startswith("⬆"), "skill"]:
    update_skill(_name, apply=False)
else:
    print("(No skills currently show an available update.)" if report["status"].str.startswith("⬆").sum() == 0 else "")